# 02. Collaborative Filtering

개인화 상품 추천 엔진 프로젝트의 Day 2 산출물입니다.  
이 노트북에서는 Memory-Based / Model-Based / ALS 기반 Collaborative Filtering 실험을 정리하고, 베이스라인 대비 성능을 비교합니다.

## ?????
- [x] Memory-Based User-CF ??
- [x] Memory-Based Item-CF ??
- [x] ??? ?? ?? (cosine / pearson / jaccard)
- [x] ?? ?(K) ??
- [x] Surprise ?? SVD / SVD++ / NMF ??
- [x] implicit ?? ALS ??
- [x] RMSE ? Precision@10 ??
- [x] ????? ?? ??? ??

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.config import DEFAULT_TOP_K, HIGH_RATING_THRESHOLD, RANDOM_SEED, TOP_K_CANDIDATES
from src.data import (
    build_relevance_sets,
    filter_seen_items,
    load_bundle,
    random_train_test_split,
    time_based_train_test_split,
)
from src.evaluation import evaluate_recommendations
from src.models import (
    baseline_summary,
    compute_similarity,
    fit_implicit_als,
    fit_surprise_model,
    predict_global_mean,
    predict_from_score_matrix,
    predict_item_based_scores,
    predict_surprise_frame,
    predict_user_based_scores,
    predict_user_mean,
    recommend_als,
    recommend_from_score_matrix,
    recommend_popular_items,
    recommend_surprise_top_k,
)

import src.models as model_hub

sns.set_theme(style="whitegrid")
np.random.seed(RANDOM_SEED)


## 1. 데이터 로드 및 평가 split 준비
Day 1에서 정의한 helper를 그대로 재사용합니다.  
실험 비교를 위해 random split과 time-based split을 모두 확인할 수 있도록 셀을 구성합니다.

In [ ]:
# bundle = load_bundle(download_if_missing=True)  # 네트워크가 가능한 환경에서만 사용
bundle = load_bundle(download_if_missing=False)
ratings = bundle.ratings

train_random, test_random = random_train_test_split(ratings, test_size=0.2)
train_time, test_time = time_based_train_test_split(ratings, test_ratio=0.2)

# Day 2 기본 실험 split: random split
train_df = train_random.copy()
test_df = test_random.copy()

pd.DataFrame({
    "split": ["train_random", "test_random", "train_time", "test_time"],
    "rows": [len(train_random), len(test_random), len(train_time), len(test_time)],
})

In [ ]:
relevant_items = build_relevance_sets(test_df, min_rating=HIGH_RATING_THRESHOLD)
catalog = sorted(train_df["item_id"].unique())
baseline_info = baseline_summary(train_df)
pd.Series(baseline_info)

## 2. 공통 평가 helper
모든 CF 모델은 같은 방식으로 Top-K 추천 품질을 평가합니다.  
중요: 추천 리스트는 반드시 train에서 본 아이템을 제외한 뒤 평가합니다.

In [ ]:
def rmse_from_frame(predictions: pd.DataFrame) -> float:
    required = {"rating", "prediction"}
    missing = required - set(predictions.columns)
    if missing:
        raise ValueError(f"Missing columns for RMSE: {missing}")
    error = predictions["rating"] - predictions["prediction"]
    return float(np.sqrt(np.mean(np.square(error))))


def evaluate_topk_model(model_name: str, recommendations: dict[int, list[int]], k: int = DEFAULT_TOP_K):
    filtered = filter_seen_items(recommendations, train_df)
    summary, user_rows = evaluate_recommendations(
        recommendations=filtered,
        ground_truth=relevant_items,
        k=k,
        catalog=catalog,
        return_user_metrics=True,
    )
    summary["model_name"] = model_name
    summary["k"] = k
    return summary, pd.DataFrame(user_rows)


## 3. 베이스라인 점검
CF 결과와 비교할 기준선으로 global mean / user mean / popularity 기반 성능을 먼저 확보합니다.

In [ ]:
baseline_results = []

global_mean_predictions = predict_global_mean(train_df, test_df)
user_mean_predictions = predict_user_mean(train_df, test_df)
popularity_recommendations = recommend_popular_items(
    train_df,
    user_ids=test_df["user_id"].unique(),
    top_k=DEFAULT_TOP_K,
)

baseline_results.append({
    "model_name": "global_mean",
    "rmse": rmse_from_frame(global_mean_predictions),
})
baseline_results.append({
    "model_name": "user_mean",
    "rmse": rmse_from_frame(user_mean_predictions),
})

popularity_summary, popularity_user_rows = evaluate_topk_model("popularity", popularity_recommendations, k=DEFAULT_TOP_K)
baseline_results.append(popularity_summary)

pd.DataFrame(baseline_results)

## 4. 사용 가능한 collaborative helper 점검
협업 필터링 구현은 `src.models`에서 병렬 개발 중일 수 있으므로, 현재 export 상태를 먼저 확인합니다.

In [ ]:
available_model_exports = sorted(
    name
    for name in dir(model_hub)
    if any(keyword in name.lower() for keyword in ["cf", "knn", "svd", "nmf", "als", "recommend"])
)
available_model_exports

## 5. Memory-Based CF 실험 (User-CF / Item-CF)
현재는 `src.models.collaborative` helper를 바로 연결할 수 있습니다.  
유사도 함수(cosine / pearson / jaccard)와 neighbor 수를 함께 비교합니다.

In [ ]:
memory_cf_grid = {
    "similarity": ["cosine", "pearson", "jaccard"],
    "neighbors": [10, 20, 40, 80],
}
memory_cf_grid

In [ ]:
memory_results = []
target_user_ids = sorted(test_df["user_id"].unique())
global_fallback = float(train_df["rating"].mean())

for similarity_name in memory_cf_grid["similarity"]:
    for n_neighbors in memory_cf_grid["neighbors"]:
        user_score_matrix, user_similarity = predict_user_based_scores(
            train_df,
            k=n_neighbors,
            metric=similarity_name,
        )
        item_score_matrix, item_similarity = predict_item_based_scores(
            train_df,
            k=n_neighbors,
            metric=similarity_name,
        )

        user_prediction_df = predict_from_score_matrix(
            user_score_matrix, test_df, default_prediction=global_fallback, model_name=f"user_cf_{similarity_name}_k{n_neighbors}"
        )
        item_prediction_df = predict_from_score_matrix(
            item_score_matrix, test_df, default_prediction=global_fallback, model_name=f"item_cf_{similarity_name}_k{n_neighbors}"
        )

        user_recommendations = recommend_from_score_matrix(
            user_score_matrix,
            train_df,
            user_ids=target_user_ids,
            top_k=DEFAULT_TOP_K,
        )
        item_recommendations = recommend_from_score_matrix(
            item_score_matrix,
            train_df,
            user_ids=target_user_ids,
            top_k=DEFAULT_TOP_K,
        )

        user_summary, _ = evaluate_topk_model(
            f"user_cf_{similarity_name}_k{n_neighbors}",
            user_recommendations,
            k=DEFAULT_TOP_K,
        )
        item_summary, _ = evaluate_topk_model(
            f"item_cf_{similarity_name}_k{n_neighbors}",
            item_recommendations,
            k=DEFAULT_TOP_K,
        )

        user_summary["rmse"] = rmse_from_frame(user_prediction_df)
        item_summary["rmse"] = rmse_from_frame(item_prediction_df)
        memory_results.extend([user_summary, item_summary])

memory_results_df = pd.DataFrame(memory_results)
memory_results_df.sort_values(by=f"precision@{DEFAULT_TOP_K}", ascending=False).head(10)


## 6. Model-Based CF (Surprise: SVD / SVD++ / NMF)
Surprise 라이브러리는 아직 설치되지 않았을 수 있으므로 optional import로 처리합니다.

In [ ]:
import subprocess

results_path = PROJECT_ROOT / 'artifacts' / 'metrics' / 'day2_collaborative_filtering_results.csv'
if not results_path.exists():
    subprocess.run([sys.executable, str(PROJECT_ROOT / 'scripts' / 'generate_day2_artifacts.py')], check=True)

comparison_df = pd.read_csv(results_path)
model_cf_df = comparison_df.loc[comparison_df['family'] == 'model_cf'].sort_values(
    by=[f'precision@{DEFAULT_TOP_K}', 'rmse'], ascending=[False, True]
)
model_cf_df.head(10)

In [ ]:
best_model_cf = model_cf_df.iloc[0][['model_name', 'rmse', f'precision@{DEFAULT_TOP_K}', f'ndcg@{DEFAULT_TOP_K}', 'coverage']]
best_model_cf

## 7. Implicit ALS 실험
ALS는 implicit 피드백 기반 실험으로 분리해 기록합니다.

In [ ]:
als_df = comparison_df.loc[comparison_df['family'] == 'implicit_als'].sort_values(
    by=f'precision@{DEFAULT_TOP_K}', ascending=False
)
als_df

In [ ]:
als_df.head(2)[['model_name', f'precision@{DEFAULT_TOP_K}', f'recall@{DEFAULT_TOP_K}', 'coverage']]

## 8. 모델 비교 테이블
모든 CF 결과를 동일한 schema로 모아서 최종 비교표를 작성합니다.

In [ ]:
comparison_columns = [
    'model_name',
    'family',
    'rmse',
    f'precision@{DEFAULT_TOP_K}',
    f'recall@{DEFAULT_TOP_K}',
    f'ndcg@{DEFAULT_TOP_K}',
    f'map@{DEFAULT_TOP_K}',
    'coverage',
]
comparison_df = comparison_df.reindex(columns=comparison_columns)
comparison_df.sort_values(by=[f'precision@{DEFAULT_TOP_K}', 'rmse'], ascending=[False, True], na_position='last').head(15)

In [ ]:
plot_df = comparison_df.dropna(subset=[f'precision@{DEFAULT_TOP_K}']).head(12).copy()
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.barplot(data=plot_df, x='model_name', y=f'precision@{DEFAULT_TOP_K}', hue='family', dodge=False, ax=axes[0])
axes[0].set_title('Top Precision@10 models')
axes[0].tick_params(axis='x', rotation=75)
axes[0].legend(loc='lower right')

rmse_plot_df = comparison_df.dropna(subset=['rmse']).sort_values('rmse').head(10)
sns.barplot(data=rmse_plot_df, x='model_name', y='rmse', hue='family', dodge=False, ax=axes[1])
axes[1].set_title('Lowest RMSE models')
axes[1].tick_params(axis='x', rotation=75)
axes[1].legend(loc='upper right')

plt.tight_layout()
plt.show()

## 9. 실험 로그 저장
모델 비교 결과는 재사용 가능하도록 `artifacts/metrics/`에 저장하는 것을 권장합니다.

In [ ]:
top10_results_path = PROJECT_ROOT / 'artifacts' / 'metrics' / 'day2_collaborative_filtering_top10.csv'
summary_paths = pd.DataFrame(
    {
        'artifact': ['full comparison', 'top10 leaderboard'],
        'path': [str(results_path.relative_to(PROJECT_ROOT)), str(top10_results_path.relative_to(PROJECT_ROOT))],
    }
)
summary_paths

## ??
- **Memory-based User-CF**? Precision@10 ???? ?? ???, ?? `pearson + k=40~80` ??? ???? ??????.
- **Surprise ??(SVD / SVD++ / NMF)**? RMSE? ? ???? Top-10 ranking ??? memory CF?? ?? ??????.
- **Implicit ALS**? ???? memory CF?? ??? Coverage? ??, long-tail ?? ????? ?? ?? ??????.
- ??? ? ??????? **?? ?? ??? ??**? **?? ??? ??**? ???? ???? ?? ?????.